# Preprocessing - coding
\ 
In this chapter we will explore in more detail the preprocessing steps used in extracellular electrophysiology. The purpose of preprocessing is to take a potentially
'noisy' recording and apply various processing steps to reduce the noise and isolate the signal. In our case, the signals of interset are recorded action potentials
and the noise sources include electrical (e.g. XX, XX), thermal (??) and kinetic (e.g. physical movement of the probe in the brain).

This chapter will focus on the practical implementation of preprocessing steps using SpikeInterface, and the importance of visually checking the results.
See the [previous chapter]() for details on the underlying theory.

In this sesssion, we will add additional useful preprocessing steps as well as inspect the effects of other common preprocesing steps. 
The steps we will explore are:

- **bandpass filtering**: Filters out low-frency oscilations (e.g. 50-60 Hz noise) and high-frequency oscilations (e.g. noise spikes) from the data. Applied separately per channel.

- **bad channel detection** : Automatically detect channels that are dead, noisy or out-of-brain. These bad channels can be removed of interpolated (i.e filled in) with the data from surrounding channels.

- **common median referencing**: Removes noise that takes the form of strips through the data (i.e. occuring on all channels for a short time period). This is applied across all channels, separately per time point.

- **highpass spatial filter**: A destriping algorithm (i.e. similar to common median referencing) introduced by the [IBL]() that accounts for slow oscilations of the signal across channels within the noise stripe.

- **motion correction** : A step to account for physical drift of the probe (typically vertical) in the brain. Attemps to estimate the drift and interpolate the channels to account for it.

- **whiten** : Removes correlated noise across channels and helps isolate spikes on  a subset of channels. Has large effects of the data scaling and waveform shape.

We will run this on a new dataset recorded on a 64-channel Cambridge Neurotech H5 probe. 
Note that we will not apply the 'phase shift' step that we did in the introduction, as that step is only relevant for Neuropixel probes.
See XXX page for information on the dataset and how to download it.

# Downloading the data

<include some information on the experiment, probably do this on a central 'dataset' page>

In this section we will download the data we will use for the majority of the rest of the course. This dataset is a XXX.

SpikeInterface has a lot of useful utilities for downloading and saving data from public repositories. This is very useful to know, because it opens up an entire world of analysis! With the code below, we can download any of the XXXX recordings on DANDI, including the entire IBL Brain Wide Map! (as specified by hte DANDI ID).

In [ ]:
import spikeinterface as si

from pathlib import Path
import numpy as np
from spikeinterface.extractors import NwbRecordingExtractor

s3_url = "https://dandiarchive.s3.amazonaws.com/blobs/a8f/800/a8f8003e-4483-4b50-8a45-91ac5971f5d5"
DURATION_S = 90
LOCAL_FOLDER = Path("_tmp/nwb_first_slice")

if LOCAL_FOLDER.exists():
    recording = si.load(LOCAL_FOLDER)
else:
    series_paths = NwbRecordingExtractor.fetch_available_electrical_series_paths(
            file_path=s3_url,
            stream_mode="remfile",
        )

    # TODO: Explain the series (dataset page)
    for p in series_paths:
        print("  -", p)

    non_lfp = [p for p in series_paths if "lfp" not in p.lower()]
    electrical_series_path = (non_lfp or series_paths)[0]

    # stream_mode="remfile" (or "fsspec") reads bytes lazily over HTTP — no full download.
    rec = NwbRecordingExtractor(
        file_path=s3_url,
        stream_mode="remfile",
        electrical_series_path=electrical_series_path,
    )

    end_frame = np.round(DURATION_S * rec.get_sampling_frequency()).astype(int)
    rec_slice_remote = rec.frame_slice(start_frame=0, end_frame=end_frame)

    recording = rec_slice_remote.save(
        format="binary",
        folder=LOCAL_FOLDER,
        n_jobs=1,
        chunk_duration="1s",
        progress_bar=True,
        overwrite=True
    )


In future, to avoid redownloading the data each time, we can download from disk

```python
if LOCAL_FOLDER.exists()
    recording = si.load(LOCAL_FOLDER)
else:
    # download the data...
```

use a dropdown here
Note we can retrieve the nwb file in code given the daandiset with the following code block (see main_nwb.py joe)
DANDISET_ID = "000939"  

# Initial steps

Before we start, we will define a function to plot and compare multiple preprocessing steps. We can reuse this function through the rest of the tutorial.
Also, feel free to instead visualise with [viewephys](https://github.com/int-brain-lab/viewephys) as we did in the [previous tutorial]().

In [ ]:
import spikeinterface.preprocessing as si_prepro
import spikeinterface.widgets as si_widgets
import matplotlib.pyplot as plt

def plot_prepro(recordings: dict, time_range: tuple[float]) -> None:
    """
    Plot multiple recordings in subplots.

    Parameters
    ----------
    recordings :
        A dictionary of recordings. The recordings will be plot and
        the corresponding keys used as plot titles.
    time_range: 
        A tuple with the (start, stop) times of the data to plot.
    """
    fig, axes = plt.subplots(
        1, len(recordings), figsize=(6 * len(recordings), 5),
        squeeze=False, layout="constrained",
    )
    for ax, (name, recording) in zip(axes.ravel(), recordings.items()):
        si_widgets.plot_traces(
            recording,
            time_range=time_range,
            mode="map",
            ax=ax,
            with_colorbar=True,
        )
        ax.set_title(name)

First, let's have a look at the effect of bandpass filtering on the data. We will plot the first 5 seconds of the data.


In [ ]:
import spikeinterface.preprocessing as si_prepro

recording_bp = si_prepro.bandpass_filter(
    recording, 
    freq_min=300, # commonly used values used based on AP waveform kinetics
    freq_max=6000
) 

plot_prepro(
    {"raw": recording, "filtered": recording_bp}, 
    time_range=(0, 5)
)

The filtering operation removes a lot of contaminating low-frequency noise, realising a significant improvement to the data. 
Plotting a shorter time window demonstrates how this improves resolution of individual spikes:

In [ ]:
TIME_RANGE = (49, 49.02)

plot_prepro(
    {"raw": recording, "filtered": recording_bp}, 
    time_range=TIME_RANGE
)

# Bad Channel Detection

Individual contacts on a probe are liable to break and become 'dead' (no signal), 'noisy' (excessive random fluctuations)
or may be 'out of brain' (i.e. a contigious set of channels at the end of the probe with no signal). 

The [IBL have developed an algorithm]() of automatically detecting bad channels, which has been ported to SpikeInterface.

First, let's apply the bad channel detection to our data and look at the data before and afterwards:

In [ ]:
bad_channel_ids_1, channel_labels_1 = si_prepro.detect_bad_channels(recording_bp)
print("first attempt labels:\n", channel_labels_1)

rec_clean_1 = recording_bp.remove_channels(bad_channel_ids_1)

plot_prepro({"filtered": recording_bp, "bad channels removed": rec_clean_1}, TIME_RANGE)

**It didn't work!** We can still see the dead channels on our recording. Why is this? 

It turns out that the IBL algorithm was developed for Neurpixel probes
(NP1 probes) but we are using Cambridge Neurotech 64-channel probes. In this case, the default settings
are not approrpiate for our data, and we need to adjust them slightly:

In [ ]:
# Let's try again with different settings
bad_channel_ids, channel_labels = si_prepro.detect_bad_channels(recording_bp, dead_channel_threshold=-0.25)
print("second attempt labels:\n", channel_labels)

rec_clean = recording_bp.remove_channels(bad_channel_ids)

plot_prepro({"filtered": recording_bp, "bad channels removed": rec_clean}, TIME_RANGE)

::: {.callout-warning}
This example demonstrates why it is critically important to **always visualise your data**
to ensure the default arguments of the algorithms we are using are suitable for your recordings.
:::


# Common average referencing

Common referencing takes a central tendency measurement (e.g. mean or median) across all channels at a single timepoint and subtracts it from the data.

This has the effect of removing noise shared across all channels, and is sometimes called 'detriping' as it removes vertical 'stripes' across channels on the heatmap. **is this true?**

We want bad channels already removed, so they do not contaminate the computaion of the mean or median. SpikeInterface provides many options for 
applying common referning to subsets of channels, and/or using particular channels as a reference. In this example we will use the default
option with `operator="median"`, which computes and subtraces the median from all channels in the recording:

In [ ]:
recording_cmr = si_prepro.common_reference(rec_clean, operator="median")

plot_prepro({"filtered": rec_clean, "recording_cmr": recording_cmr}, TIME_RANGE)

We can see that contaminating noise spikes that run vertically across the heatmap are removed through this processing method.

IBL also has the highpass spatial filter.

In [ ]:
recording_hsf = si_prepro.highpass_spatial_filter(rec_clean)

plot_prepro({"recording_cmr": recording_cmr, "recording_hsf": recording_hsf}, TIME_RANGE)

recording_hsf = si_prepro.highpass_spatial_filter(rec_clean, highpass_butter_wn=0.01)

plot_prepro({"recording_cmr": recording_cmr, "recording_hsf": recording_hsf}, TIME_RANGE)

recording_hsf = si_prepro.highpass_spatial_filter(rec_clean, highpass_butter_wn=0.02)

plot_prepro({"recording_cmr": recording_cmr, "recording_hsf": recording_hsf}, TIME_RANGE)

recording_hsf = si_prepro.highpass_spatial_filter(rec_clean, highpass_butter_wn=0.06)

plot_prepro({"recording_cmr": recording_cmr, "recording_hsf": recording_hsf}, TIME_RANGE)

recording_hsf = si_prepro.highpass_spatial_filter(rec_clean, highpass_butter_wn=0.1)

plot_prepro({"recording_cmr": recording_cmr, "recording_hsf": recording_hsf}, TIME_RANGE)

recording_hsf = si_prepro.highpass_spatial_filter(rec_clean, highpass_butter_wn=0.25)

plot_prepro({"recording_cmr": recording_cmr, "recording_hsf": recording_hsf}, TIME_RANGE)


:::{.callout-tip}
### Things to try

- The time window we plot HAS IMPORTANT. Each spike is around 1-5 ms long, and so if we plot long time windows we don't have enough pixels on the screen to resolve spikes. Nonetheless this can reveal other patterns in the data (e.g. 50Hz noise). Play around with plotting different time windows (e.g. 10ms, 200 ms, 1s) etc. to get a feel for this.
- Plot the data using the `mode=line` argument. Same data - a completely different perspective!
- Play around with the filter cutoffs in bandpass filter. What happens if we set the XX very XXX or the XXX very XXX What happens if you set the bandpass filter minimum cutoff to zero? What frequencies are including now? Or set the maximum cutoff to 15000?
\
- `return_in_uV=False`. Does the data look much different? What is the datatype now? Can you explain what has happened to the data? Use `.get_traces()`
- CMR with special arguments
- HSF?
\
- Compute the Fourier transform of a raw data channel and plot the frequency spectrum
- Perform spike detection (e.g. absolute threshold method) on the preprocessed data obtained with `get_traces()`

:::

# Motion correction

Motion c

In [ ]:
rec_corrected, motion_info = si_prepro.correct_motion(
    recording_cmr,
    preset="dredge_fast",
    output_motion_info=True,
)
print(rec_corrected)

More stuff

In [ ]:
# Display the motion output: drift map + estimated motion over depth/time.
fig = plt.figure(figsize=(14, 8))
si_widgets.plot_motion_info(
    motion_info,
    recording=rec_corrected,
    figure=fig,
    color_amplitude=True,
    amplitude_cmap="inferno",
    scatter_decimate=10,
)
fig.suptitle("Motion correction output")

because its a small recording, dont see much. We can see in longer recordings, the drift correction is much more obvious:

Explain the image, then add the https://spikeinterface.readthedocs.io/en/stable/how_to/handle_drift.html#handle-drift-in-your-recording

explain the spiking phenomenon!?

# Whitening

Finally, we will explore the whiteinng step. The whitening step is an agressive step to remove noise correlations and xx across channels.
Has the effect of removing scaled versions of neighoubring races h3h3h3

In spikeinterface, whitening is done by the `whiten` function:

In [ ]:
recording_whiten = si_prepro.whiten(rec_corrected)

plot_prepro({"rec_corrected": rec_corrected, "whitened": recording_whiten}, TIME_RANGE)

For visualisation, let's rescale hte recroding. **Note we would not do this in our actual analysis, this is just for visualising the data as part of this course**

In [ ]:
# Choose some example times that show the phenomenon well
start_frame = int(51.475 * rec_corrected.get_sampling_frequency())
end_frame = int(51.55 * rec_corrected.get_sampling_frequency())

rec_range = np.ptp(
    rec_corrected.get_traces(start_frame=start_frame, end_frame=end_frame, return_in_uV=True)
)  

whitened_range = np.ptp(
    recording_whiten.get_traces(start_frame=start_frame, end_frame=end_frame, return_in_uV=True)
)  
estimate_scale_difference = rec_range / whitened_range

print(estimate_scale_difference)
scaled_white_recording = si_prepro.scale(recording_whiten, estimate_scale_difference)

plot_prepro({"rec_corrected": rec_corrected, "scaled_white_recording": scaled_white_recording}, (51.475, 51.55))

We can see a few of the effects. Let's also plot the 
Whats going on here? Whitening scales the recording

In [ ]:
first_data = rec_corrected.get_traces(start_frame=start_frame, end_frame=end_frame, return_in_uV=True)
third_data = scaled_white_recording.get_traces(start_frame=start_frame, end_frame=end_frame)

# We add the offset with np.arange so the traces are vetically separated
plt.plot(first_data + np.arange(first_data.shape[1])*500, color="k")
plt.plot(third_data + np.arange(third_data.shape[1])*500, color="r")
plt.xlabel("sample number)")

# Turn off the steps in KS! 

We must remember to turn off! Note kS does opposite order, we can't necessary be sure internal assumptions of KS are
met when we perform preprocessing outside of it. Make sure to carefully check the sorting results.

Or use other sorters!


In [ ]:
#| eval: false
import spikeinterface.sorters as si_sorters

sorting = si_sorters.run_sorter(
    "kilosort4",
    recording_whiten,
    folder="_tmp/ks4_out",
    remove_existing_folder=True,
    verbose=True,
    skip_kilosort_preprocessing=True,
    do_correction=False,
)
print(sorting)